# Module 1: Data Prep

In [3]:
# Importing Libraries
import numpy as np
import pandas as pd

In [6]:
df = pd.read_csv("DataCo_ML_Dataset.csv")

In [7]:
# looking at the first 5 rows to ensure the headers and data looking correct
display(df.head())

# checking for missing (null) values that could break our ML model
print(df.isnull().sum(), "\n")

# checking the exact shape (Number of Rows, Number of Columns)
print(f"Total Rows and Columns: {df.shape}")

,Late_Delivery_Risk,Sales,Order_Item_Quantity,Order_Profit_Per_Order,Days_Scheduled,Category_Name,Customer_Segment,Shipping_Type,Customer_Country,Order_Country,Month_Name,Day_of_Week
0,0,299.98,1,88.79,4,Camping & Hiking,Consumer,CASH,EE. UU.,MTxico,January,Thursday
1,0,199.99,1,91.18,4,Water Sports,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
2,0,250.00,5,68.25,4,Women's Apparel,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
3,0,129.99,1,36.47,4,Men's Footwear,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
4,1,49.98,2,4.10,4,Accessories,Home Office,CASH,EE. UU.,Colombia,January,Thursday


Late_Delivery_Risk        0
Sales                     0
Order_Item_Quantity       0
Order_Profit_Per_Order    0
Days_Scheduled            0
Category_Name             0
Customer_Segment          0
Shipping_Type             0
Customer_Country          0
Order_Country             0
Month_Name                0
Day_of_Week               0
dtype: int64 

Total Rows and Columns: (180519, 12)


Data Quality checking: Verified 0 missing values across all 180,000+ records. The Star Schema architecture successfully enforced data integrity during the SQL extraction phase.


In [8]:
categorical_cols = [
    "Category_Name",
    "Customer_Segment",
    "Shipping_Type",
    "Customer_Country",
    "Order_Country",
    "Month_Name",
    "Day_of_Week",
]

# 3. Applying One-Hot Encoding to translate text to 0s and 1s
df_encoded = pd.get_dummies(df, columns = categorical_cols, drop_first=True)

In [9]:
df_encoded.head() # see now it has 240 columns

,Late_Delivery_Risk,Sales,Order_Item_Quantity,Order_Profit_Per_Order,Days_Scheduled,Category_Name_As Seen on TV!,Category_Name_Baby,Category_Name_Baseball & Softball,Category_Name_Basketball,Category_Name_Books,...,Month_Name_May,Month_Name_November,Month_Name_October,Month_Name_September,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
0,0,299.98,1,88.79,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,0,199.99,1,91.18,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,0,250.00,5,68.25,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,0,129.99,1,36.47,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,1,49.98,2,4.10,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


## Module 2: The Train-Test Split

### USING Scikit-Learn (sklearn)

In [10]:
# Importing the splitting tool from Scikit-Learn
from sklearn.model_selection import train_test_split

# 1. Separating the Target (y) from the Features (X)
# 'y' is the Final Answer we want to predict

y = df_encoded['Late_Delivery_Risk']

x = df_encoded.drop('Late_Delivery_Risk', axis = 1)

X_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state=42)

print(df_encoded.shape)
print(X_train.shape)
print(x_test.shape)

(180519, 240)
(144415, 239)
(36104, 239)


### Module 3: Feature Scaling

- StandardScaler

In [11]:
# Importing the scaling tool from Scikit-Learn
from sklearn.preprocessing import StandardScaler

# 1. Creating the scaler
scaler = StandardScaler()

# 2. We only want to scale our continuous numbers! 
num_cols = ['Sales', 'Order_Item_Quantity', 'Order_Profit_Per_Order', 'Days_Scheduled']

# 3. Applying the scaler to our Training and Testing data
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

X_train[num_cols].head()

print()

### Module 4: The Machine Learning Model (Random Forest)
For this supply chain problem, we need the AI to predict a simple "Yes" or "No" (1 or 0): Will this order be late?

- The Random Forest Classifier.

In [12]:
# Import the Random Forest algorithm and our grading tools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [13]:
# 1. Bring in the AI
# n_estimators=100 means we are hiring 100 Decision Trees to vote on the answer
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Training the AI
print("Training the AI... (Grab a sip of coffee, this might take a minute!)")
rf_model.fit(X_train, y_train)
print("Training Complete! The AI has learned the patterns.")

Training the AI... (Grab a sip of coffee, this might take a minute!)


Training Complete! The AI has learned the patterns.


In [14]:
print("Taking the Final Exam...")
y_pred = rf_model.predict(x_test)
print('done')

Taking the Final Exam...
done


In [15]:
# 4. Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🏆 Accuracy): {accuracy * 100:.2f}%")


🏆 Accuracy): 73.31%


In [16]:
# 5. The Detailed 
print("\n Classification Report:")
print(classification_report(y_test, y_pred))


 Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.77      0.72     16273
           1       0.79      0.70      0.74     19831

    accuracy                           0.73     36104
   macro avg       0.73      0.74      0.73     36104
weighted avg       0.74      0.73      0.73     36104



In [17]:
# Import the saving tool
import joblib

joblib.dump(rf_model, 'dataco_rf_model.joblib')

joblib.dump(scaler, 'dataco_scaler.joblib')

joblib.dump(X_train.columns, 'dataco_columns.joblib')

print("AI Brain, Scaler, and Columns successfully saved to the hard drive!")

AI Brain, Scaler, and Columns successfully saved to the hard drive!


In [24]:
import joblib

# The compress=3 parameter shrinks the file size massively!
joblib.dump(rf_model, 'dataco_rf_model.joblib', compress=8)

['dataco_rf_model.joblib']